In [3]:
import pandas as pd
import os


In [4]:
basePath = '5YearSpanDatasets'
forces = ['metropolitan', 'thames', 'surrey', 'sussex']

allData = []

# Checks each folder in the updatedDatasets folder
for folder in os.listdir(basePath):
    # Concatenates the folder name to the updatedDatasets to create a new file path
    filePath = os.path.join(basePath, folder)
    # Checks if the folder actually exists
    if os.path.isdir(filePath):
        # Checks each file in the file path
        for file in os.listdir(filePath):
            # Checks the file contains one of the police forces in the filename
            if any(force in file.lower() for force in forces):
                # Updates file path to include the folder
               file_path = os.path.join(filePath, file)
               
               df=pd.read_csv(file_path)
               df['Month'] = folder
               allData.append(df)
    

               


In [5]:
if allData:
    masterDf = pd.concat(allData, ignore_index=True)
else:
    print("No data found to process")

In [6]:
# Shows number of records in the master data
print(f"Before preprocessing, there are {len(masterDf)} values in the data")

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

# Shows the unique values for each column
for col in masterDf:
    print(f" - The unique values for {col} are {masterDf[col].unique()}")


Before preprocessing, there are 9453125 values in the data

List of null values in every column 
 Crime ID                 2111038
Month                          0
Reported by                    0
Falls within                   0
Longitude                 176614
Latitude                  176614
Location                       0
LSOA code                 176644
LSOA name                 176644
Crime type                     0
Last outcome category    2111038
Context                  9453125
dtype: int64

 - The unique values for Crime ID are ['a8977a2a4e14252420371eb993d52e4d0b8288a7c833e66ba79592b3e2b982e7' nan
 '934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c3eee2c73f16630edc5' ...
 'cb80080e0164fa9d12767bcfe912ed902b1bc3db0ce83d50b7f6bc055d313392'
 'c49ac623430308ffb6f815e12725a9d3244c7bf3275592a22953ed4f5964c326'
 '14fb64f2b5de925c7c1cec2234f64d221dada2af05b6bc07fe41db9139e021ea']
 - The unique values for Month are ['2019-01' '2019-02' '2019-03' '2019-04' '2019-05' '2019-06' '2019-07'

In [8]:
totalBeforeDupRemoval = len(masterDf)
print(f"There are {totalBeforeDupRemoval} values in the dataset before any processing at all")
duplicateCount = masterDf.duplicated().sum()
print(f"There are {duplicateCount} duplicate rows.")
masterDf = masterDf.drop_duplicates()
totalAfterDupRemoval = len(masterDf)
print(f"There are {totalAfterDupRemoval} values in the dataset after removing duplicates")

There are 8441407 values in the dataset before any processing at all
There are 0 duplicate rows.
There are 8441407 values in the dataset after removing duplicates


In [9]:
# Handling Missing Values
# Missing values in Crime ID , Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category, Context
# Context is missing for every single record so can be dropped

print(f"There are {totalAfterDupRemoval} total records")

# Shows each column and if they have any null values
print(f"\nList of null values in every column before handling null values\n {masterDf.isnull().sum()}\n")

# Crime ID: add placeholder value "No Crime ID"
# Longitude and Latitude: add placeholder value 0
# LSOA code: add placeholder value "No LSOA Code Assigned"
# LSOA name: add placeholder value "No LSOA name Assigned"
# Last outcome category: add placeholder value "Unknown outcome category"
# Context: drop since every record has null values in context

placeholders = {
    'Crime ID': 'No Crime ID',
    'Longitude': 0,
    'Latitude' : 0,
    'LSOA code' : 'No LSOA Code assigned',
    'LSOA name' : 'No LSOA Name assigned',
    'Last outcome category' : 'Unknown outcome category'
}

masterDf = masterDf.fillna(value=placeholders)
masterDf = masterDf.drop("Context", axis='columns')

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

There are 8441407 total records

List of null values in every column before handling null values
 Crime ID                 1155292
Month                          0
Reported by                    0
Falls within                   0
Longitude                 167326
Latitude                  167326
Location                       0
LSOA code                 167356
LSOA name                 167356
Crime type                     0
Last outcome category    1155292
Context                  8441407
dtype: int64


List of null values in every column 
 Crime ID                 0
Month                    0
Reported by              0
Falls within             0
Longitude                0
Latitude                 0
Location                 0
LSOA code                0
LSOA name                0
Crime type               0
Last outcome category    0
dtype: int64



In [ ]:
# Dropping "Falls within" column
masterDf = masterDf.drop("Falls within", axis='columns')

for col in masterDf:
    print(col)

Crime ID
Month
Reported by
Longitude
Latitude
Location
LSOA code
LSOA name
Crime type
Last outcome category


In [11]:
# Standardize 'Crime type' and 'Last outcome category'
masterDf['Crime type'] = masterDf['Crime type'].str.strip().str.capitalize()
masterDf['Last outcome category'] = masterDf['Last outcome category'].str.strip().str.capitalize()

In [13]:
masterDf['MonthDateTime'] = pd.to_datetime(masterDf['Month'])
masterDf['Year'] = masterDf['MonthDateTime'].dt.year

In [34]:
xls = pd.ExcelFile('policeforceareas1991to2024.xlsx')
pop_11_20 = pd.read_excel(xls, 'Mid-2011 to Mid-2020', skiprows=3)
pop_21_24 = pd.read_excel(xls, 'Mid-2021 to Mid-2024', skiprows=3)

popAll = pd.concat([pop_11_20, pop_21_24], axis=0)
populationColumns = popAll.columns[3:]
popAll['Population'] = popAll[populationColumns].sum(axis=1)

popClean = popAll[['PFA 2023 Name', 'Year', 'Population']].copy()
popClean.columns = ['Force', 'Year', 'Population']
popClean = popClean.reset_index(drop=True)

print(popClean.head())





                 Force  Year  Population
0  Metropolitan Police  2011     8196995
1  Metropolitan Police  2012     8313253
2  Metropolitan Police  2013     8431450
3  Metropolitan Police  2014     8539625
4  Metropolitan Police  2015     8651877


In [40]:
popClean['Force'] = popClean['Force'].str.replace("Police", "").str.strip()

print(popClean.head())


          Force  Year  Population
0  Metropolitan  2011     8196995
1  Metropolitan  2012     8313253
2  Metropolitan  2013     8431450
3  Metropolitan  2014     8539625
4  Metropolitan  2015     8651877
